# 🚀 Stack 2.9 - Colab Training Notebook

**Zero-cost training on Google Colab free tier**

This notebook trains a LoRA adapter for Stack 2.9 Pattern Memory on **Qwen2.5-Coder-7B** using a free T4 GPU.

⏱️ **Expected runtime:** 3-5 hours
💾 **VRAM needed:** ~12GB (fits in T4's 15GB)
📦 **Output:** `./adapters_colab/` (LoRA) + `./model_final/` (merged)

---

**Instructions:**
1. Runtime → Change runtime type → **GPU (T4)**
2. Run each cell in order (Shift+Enter or Play button)
3. Monitor progress in cell outputs

---

In [1]:
# Check GPU availability
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## 1️⃣ Mount Google Drive (Optional)

Mount Drive to persist data between sessions. If you skip this, data is stored in Colab's ephemeral storage (lost after ~12h or disconnect).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set base path (change if not using Drive)
BASE_PATH = "/content/drive/MyDrive/stack-2.9"  # or use "/content/stack-2.9" for local storage

print(f"Using base path: {BASE_PATH}")

## 2️⃣ Clone Repository & Install Dependencies

In [ ]:
import os
os.chdir('/content')

# Clone the Stack 2.9 repository if not already present
if not os.path.exists('stack-2.9'):
    !git clone https://github.com/my-ai-stack/stack-2.9.git

os.chdir('/content/stack-2.9')

# Upgrade pip and install dependencies
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.40.0 peft==0.10.0 accelerate bitsandbytes==0.43.0 datasets pyyaml

## 3️⃣ Prepare Training Data

### Option A: Use existing training data from repository
The repo already has `training-data/final/train.jsonl` and `eval.jsonl` if you previously ran data collection.

### Option B: Create a mini dataset for quick prototyping (5K examples)
Recommended for first run to verify everything works quickly.

In [ ]:
# Create mini dataset (5K examples)
!python scripts/create_mini_dataset.py --size 5000 --output data_mini/train_mini.jsonl --source training-data/final/train.jsonl

# Check what we have
!ls -lh data_mini/

# If you want to use the full dataset instead, skip the mini creation and use:
# training-data/final/train.jsonl (and eval.jsonl if available)

## 4️⃣ Prepare Training Configuration

Copy the Colab-optimized config (7B model, 8K seq len, 2 epochs).

In [ ]:
# Copy the Colab config or create one manually
!cp stack_2_9_training/train_config_colab.yaml stack_2_9_training/train_config.yaml

# If you need to use your own data, edit the paths in train_config.yaml:
# data:
#   train_file: "./data_mini/train_mini.jsonl"  # or ./training-data/final/train.jsonl
#   validation_file: "./training-data/final/eval.jsonl"  # optional

print("Configuration ready. Showing relevant sections:")
with open('stack_2_9_training/train_config.yaml', 'r') as f:
    lines = f.readlines()
    for i, line in enumerate(lines[:50]):  # Show first 50 lines
        print(f"{i+1}: {line.rstrip()}")
print("...")

## 5️⃣ Train LoRA Adapter

This is the main training step. Monitor GPU memory with `nvidia-smi` in a separate terminal or cell.

⚠️ **Training will take 3-5 hours**. Do not interrupt unless necessary.

Training progress is shown with loss values. Lower loss = better learning.

In [ ]:
# Start training
%env PYTHONUNBUFFERED=1  # Force unbuffered output for real-time logs

!cd stack_2_9_training && python -m train_lora --config train_config.yaml

# Checkpoints are saved to ./adapters_colab/ every 500 steps

## 6️⃣ Verify Training Output

In [ ]:
!ls -lh adapters_colab/

# If training succeeded, you should see:
# - adapter_model.bin (or multiple checkpoint-XXX folders)
# - training_args.bin
# - config.json

## 7️⃣ Merge LoRA Adapter with Base Model

Combines the trained adapter weights with the base Qwen2.5-Coder-7B model to produce a standalone fine-tuned model.

In [ ]:
!cd stack_2_9_training && python -m merge_adapter --base-model Qwen/Qwen2.5-Coder-7B

print("\n✅ Merged model created in ./model_final/")
!ls -lh model_final/

## 8️⃣ Test Inference

Quick sanity check: does the model generate reasonable code?

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load merged model
model_path = "./model_final"
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

# Test generation
prompt = "Write a Python function to calculate factorial recursively:\n\n```python\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("Generating...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=\"*40)
print("RESPONSE:")
print("=\"*40)
print(response[len(prompt):])  # Show only generated part

## 9️⃣ Export to Hugging Face Hub (Optional)

If you want to publish your model, push it to Hugging Face and then apply to Together AI.

In [ ]:
from huggingface_hub import HfApi

# You need a Hugging Face account and token
HF_TOKEN = input("Enter your Hugging Face token: ").strip()

api = HfApi(token=HF_TOKEN)

# Choose a repo name
repo_id = input("Enter repository name (e.g., your-org/stack-2.9-7b-lora): ").strip()

print(f"\nUploading to {repo_id}...")

# Create repo if needed
api.create_repo(repo_id=repo_id, exist_ok=True)

# Upload model
api.upload_folder(
    folder_path="./model_final",
    repo_id=repo_id,
    repo_type="model"
)

print(f"\n✅ Model uploaded to https://huggingface.co/{repo_id}")

# Update docs
print("\nNext steps:")
print("1. Update TOGETHER_AI.md with your model ID")
print("2. Update README.md badges with real scores after evaluation")
print("3. Submit to Together AI model submission form")

## 🎉 Training Complete!

You now have:
- ✅ Trained LoRA adapter in `./adapters_colab/`
- ✅ Merged full model in `./model_final/`
- ✅ Model card and documentation

**Next steps:**
1. Run proper evaluation using `run_proper_evaluation.py`
2. Update README with real benchmark scores
3. Apply to Together AI with your Hugging Face model

**Need help?** See `COLAB_TRAINING.md` for detailed troubleshooting.